# Eigenvector Rotation Precedes Eigenvalue-Based Early-Warning Signals
**ArXivist-generated reproduction notebook**

Paper: [arXiv:2607.11935](https://arxiv.org/abs/2607.11935)
Generated: 2026-07-18

This notebook walks through every component of the implementation: the
TVP-Kalman elasticity filter, classical rolling-window EWS, lead-lag
analysis, and the six simulated tipping-point validation systems. No GPU is
needed — everything here is lightweight NumPy/SciPy time-series computation.

In [ ]:
# Environment check -- no GPU needed, this is pure NumPy/SciPy/pandas
import sys
print(f"Python: {sys.version}")

import numpy as np
print(f"NumPy: {np.__version__}")
import scipy
print(f"SciPy: {scipy.__version__}")
import sklearn
print(f"scikit-learn: {sklearn.__version__}")


In [ ]:
# Install the project in editable mode (run once). Falls back to
# --break-system-packages if pip refuses on an "externally managed"
# environment (common on Debian/Ubuntu Python 3.12+); prefer a virtualenv
# for a real project instead of this flag.
import subprocess

result = subprocess.run(["pip", "install", "-e", ".."], capture_output=True, text=True)
if result.returncode != 0 and "externally-managed-environment" in result.stderr:
    result = subprocess.run(
        ["pip", "install", "--break-system-packages", "-e", ".."], capture_output=True, text=True
    )
print(result.stdout[-2000:] if result.returncode == 0 else result.stderr[-2000:])


## Paper Overview

**Problem**: Classical early-warning signals (EWS) for critical transitions
rely on the dominant *eigenvalue* of the system's Jacobian — rising variance
and lag-1 autocorrelation (AR(1)). These have $O(\delta\theta^2)$
sensitivity to the distance-to-bifurcation $\delta\theta$, so detectable
signals only emerge when the system is already close to tipping.

**Core idea**: eigen*vectors* rotate with $O(\delta\theta)$ sensitivity — a
full order of magnitude faster. This paper measures that rotation via a
dimensionless time-varying elasticity:

$$\beta(t) \equiv \frac{d\log y}{d\log x}$$

estimated with a **TVP-Kalman filter** (time-varying-parameter regression via
Kalman filter) in log-log space, followed by Rauch-Tung-Striebel (RTS)
backward smoothing.

**Validation**: on 24 years of monthly NASA AIRS satellite data (three
climate regions) and six simulated systems with known tipping points, β is
shown to be (1) orthogonal to AR(1), and (2) systematically leads it.

## Component 1: The Taylor-expansion state-transition matrix

The Kalman filter tracks a 4-state vector $x_k = [\beta_k, \beta'_k, \beta''_k, \beta'''_k]^T$,
propagated by $x_k = F x_{k-1} + \eta_k$. $F$ is the standard factorial-scaled
kinematic/Taylor chain matrix (the paper names this matrix but doesn't give
numeric entries — see README.md for this assumption).

In [ ]:
import numpy as np
from ews_kalman.kalman import build_taylor_transition_matrix

F = build_taylor_transition_matrix(dt=1/12, order=3)
print("F (order-3 Taylor transition matrix, dt=1/12 year):")
print(np.round(F, 5))


## Component 2: TVP-Kalman filter — recovering a known elasticity

Sanity check: if $y = x^\beta$ exactly (no noise), the log-log filter should
recover $\beta$ almost exactly after a short settling period.

In [ ]:
from ews_kalman.kalman import TVPKalmanFilter

rng = np.random.default_rng(0)
N = 200
beta_true = 0.5
x = np.abs(100 + rng.normal(0, 5, N).cumsum() * 0.1) + 50
y = x ** beta_true

kf = TVPKalmanFilter(R=1e-6, Q_diag=(1e-8, 1e-9, 1e-10, 1e-11), dt=1.0, mode="loglog")
result = kf.estimate_beta(x, y)

print(f"True beta: {beta_true}")
print(f"Estimated beta (mean after settling, steps 20+): {np.mean(result['beta'][20:]):.4f}")
print(f"Estimated beta (final value): {result['beta'][-1]:.4f}")


## Component 3: Classical rolling-window EWS

AR(1), variance, permutation entropy, and mutual information — the
classical eigenvalue-based signals β is compared against.

In [ ]:
from ews_kalman.ews import ClassicalEWS

ews = ClassicalEWS()
rng = np.random.default_rng(1)

# White noise should give AR(1) near zero; a linear ramp should give AR(1) near 1
white_noise = rng.normal(0, 1, 100)
linear_ramp = np.linspace(0, 100, 100)

ar1_noise = ews.rolling_ar1(white_noise, window=24)
ar1_ramp = ews.rolling_ar1(linear_ramp, window=24)

print(f"AR(1) for white noise (expect ~0): {np.mean(ar1_noise):.3f}")
print(f"AR(1) for linear ramp (expect ~1): {np.mean(ar1_ramp):.3f}")

pe = ews.rolling_permutation_entropy(white_noise, embedding_dim=3, window=36)
print(f"Permutation entropy for white noise (expect high, near 1): {np.mean(pe):.3f}")


## Component 4: The NASA AIRS regional data loader (synthetic fallback)

NASA AIRS data is freely available but requires an interactive Giovanni
portal pull (see `data/README_data.md`). The synthetic fallback below
generates illustrative T, q series calibrated to roughly match the paper's
reported |β| per region (Table 1).

In [ ]:
from ews_kalman.data import AIRSDataLoader

loader = AIRSDataLoader()
tropics = loader.load_region("tropics", data_dir="/nonexistent/path", seed=0)
print(f"Tropics region: T shape={tropics['T'].shape}, q shape={tropics['q'].shape}")
print(f"First 5 T values: {tropics['T'][:5].round(2)}")


## Component 5: Full regional analysis — Table 1 reproduction

Run the TVP-Kalman filter + classical EWS on the synthetic Tropics region
and reproduce one row of Table 1 (|β|, σ_β, r(β,AR1), r(β,MI), transitions).
The Tropics region should show β close to the Clausius-Clapeyron scaling
value (~0.5) with low volatility (stable coupling) — the paper reports
|β|=0.492, σ_β=0.035.

In [ ]:
from ews_kalman.evaluation import RegionSummaryComputer

kf = TVPKalmanFilter(R=0.001, Q_diag=(1e-6, 1e-7, 1e-8, 1e-9), dt=1/12, mode="loglog")
beta_result = kf.estimate_beta(tropics["T"], tropics["q"])

ar1_T = ews.rolling_ar1(tropics["T"], window=24)
mi = ews.rolling_mutual_information(tropics["T"], tropics["q"], window=36, n_neighbors=3)

computer = RegionSummaryComputer()
row = computer.compute_table1_row(beta_result["beta"], ar1_T, mi, beta_result["beta_double_prime"])

print("Table 1 row (synthetic Tropics):")
for k, v in row.items():
    print(f"  {k}: {v}")
print()
print("Paper's reported Tropics row: N=284, |beta|=0.492, sigma_beta=0.035, "
      "r(beta,AR1_T)=+0.08 (p=0.22), r(beta,MI)=-0.33 (p<0.001), transitions=2")


## Component 6: Six simulated tipping-point systems — Table 3 reproduction

Validates the central hypothesis: β leads AR(1) specifically when the
transition involves *coupling degradation* (Stommel AMOC, critical slowing
down), but not when only a variable's own stability changes without a
coupling change (fold bifurcation).

In [ ]:
from ews_kalman.evaluation import SimulationValidator

validator = SimulationValidator()
results = validator.validate_all_systems(seed=0)

print(f"{'Simulation':<22} {'Tipping t':>10} {'beta lead':>10} {'AR1 lead':>10} {'Winner':>10}")
print("-" * 66)
for r in results:
    beta_lead_str = str(r["beta_lead"]) if r["beta_lead"] is not None else "-"
    ar1_lead_str = str(r["ar1_lead"]) if r["ar1_lead"] is not None else "-"
    print(f"{r['simulation']:<22} {r['tipping_t']:>10} {beta_lead_str:>10} {ar1_lead_str:>10} {r['winner']:>10}")

print()
print("Paper's reported Table 3 (for comparison -- exact simulation parameters")
print("are not fully specified in the paper, so exact numbers won't match):")
print("  Fold bifurcation:      AR1 wins (beta never detects)")
print("  Beta step change:      close, 'AR1' wins narrowly (165 vs 170)")
print("  Beta linear decay:     beta only detects (AR1 never crosses threshold)")
print("  Logistic map:          tie (100 vs 100)")
print("  Stommel AMOC:          beta wins by +39")
print("  Critical slowing down: beta wins by +153")


## Paper's Reported Results (for reference)

The numbers below are directly transcribed from the paper's Tables 1-3, for
comparison against your own full-scale run (`run_all.py`).

In [ ]:
paper_results = {
    "Arctic |beta| (paper Table 1)": 0.106,
    "Arctic sigma_beta": 0.139,
    "Tropics |beta|": 0.492,
    "Tropics sigma_beta": 0.035,
    "Monsoon |beta|": 0.477,
    "Monsoon sigma_beta": 0.062,
    "beta leads AR(1) by (observational, months)": "14-24",
    "beta leads AR(1) by (Stommel AMOC, timesteps)": 39,
    "beta leads AR(1) by (critical slowing down, timesteps)": 153,
}
for k, v in paper_results.items():
    print(f"  {k}: {v}")

print()
print("To reproduce the full analysis (all 3 regions + all 6 simulations):")
print("  python run_all.py --config configs/config.yaml --output-dir results/")


## What to do next

1. **Full observational analysis**: `python run_observational_analysis.py --config configs/config.yaml --region all`
2. **Full simulation validation**: `python run_simulation_validation.py --config configs/config.yaml --system all`
3. **Both, with Figures 1-3**: `python run_all.py --config configs/config.yaml --output-dir results/`

**Top 3 implementation notes to be aware of** (see `sir.json` and README.md for the full list):

1. The Kalman state-transition matrix *F* is named but not numerically specified in the paper —
   the standard factorial-scaled Taylor/kinematic-chain matrix is used (confidence 0.55).
2. The six simulated systems' noise levels and forcing schedules are not given numerically —
   illustrative parameters reproduce the *qualitative* pattern of Table 3 well, but not exact numbers.
3. NASA AIRS data requires a manual Giovanni-portal pull; a synthetic fallback (used above)
   lets the whole pipeline run without live data access, but is not real climate data.
